In [1]:
%load_ext autoreload
%autoreload 2

%cd ..

c:\Users\admin\Desktop\Matej\FER\Diplomski\3.semestar\NLP\NLP-projekt


In [58]:
import pandas as pd
import geopandas as gpd
from shapely import Point
from data.lucas_descriptions import numerical_features, non_numerical_features, nuts_to_country

In [59]:
def convert_potential_numerical(data: pd.DataFrame, numerical_columns: list[str] = []) -> pd.DataFrame:
    convertible_numeric_columns = []
    for col in data.columns:
        try:
            # Try converting the column to numeric; check for NaNs
            temp = pd.to_numeric(data[col], errors='coerce')
            if col in numerical_columns:  # Check if there are not many NaN values
                data[col] = temp
                convertible_numeric_columns.append(col)
        except:
            pass  # Ignore errors for non-convertible columns

    # Convert only those columns to numeric
    data[convertible_numeric_columns] = data[convertible_numeric_columns].apply(pd.to_numeric)
    
    return data

In [60]:
numerical_columns = numerical_features['Column Name'].values.tolist()
non_numerical_columns = non_numerical_features['Column Name'].values.tolist()

In [65]:
lucas_df = pd.read_csv("./data/LUCAS-SOIL-2018.csv")
lucas_df = convert_potential_numerical(lucas_df, numerical_columns)
lucas_df['SURVEY_DATE'] = pd.to_datetime(lucas_df['SURVEY_DATE'], format='%d-%m-%y')
lucas_df['Country'] = lucas_df['NUTS_0'].map(nuts_to_country)

geometry = [
    Point(lon, lat)
    for lat, lon in zip(lucas_df['TH_LAT'], lucas_df['TH_LONG'])
]
lucas_gdf = gpd.GeoDataFrame(lucas_df, geometry=geometry, crs="EPSG:4326")

In [66]:
lucas_gdf.to_file("data/LUCAS_SOIL_2018_processed.gpkg", driver="GPKG")

In [67]:
loaded_gdf = gpd.read_file("data/LUCAS_SOIL_2018_processed.gpkg")

In [69]:
loaded_gdf[numerical_columns].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18984 entries, 0 to 18983
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   pH_CaCl2          18983 non-null  float64
 1   pH_H2O            18983 non-null  float64
 2   EC                18975 non-null  float64
 3   OC                18949 non-null  float64
 4   CaCO3             11212 non-null  float64
 5   P                 13981 non-null  float64
 6   N                 18969 non-null  float64
 7   K                 18944 non-null  float64
 8   OC (20-30 cm)     140 non-null    float64
 9   CaCO3 (20-30 cm)  14 non-null     float64
 10  Ox_Al             2510 non-null   float64
 11  Ox_Fe             2510 non-null   float64
 12  TH_LAT            18984 non-null  float64
 13  TH_LONG           18984 non-null  float64
 14  Elev              18984 non-null  int64  
dtypes: float64(14), int64(1)
memory usage: 2.2 MB


In [70]:
loaded_gdf[non_numerical_columns].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18984 entries, 0 to 18983
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Depth        18984 non-null  object        
 1   POINTID      18984 non-null  int64         
 2   Country      18984 non-null  object        
 3   NUTS_0       18984 non-null  object        
 4   NUTS_1       18984 non-null  object        
 5   NUTS_2       18984 non-null  object        
 6   NUTS_3       18984 non-null  object        
 7   SURVEY_DATE  18984 non-null  datetime64[ns]
 8   LC           18984 non-null  object        
 9   LU           18984 non-null  object        
 10  LC0_Desc     18984 non-null  object        
 11  LC1_Desc     18984 non-null  object        
 12  LU1_Desc     18984 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(11)
memory usage: 1.9+ MB
